# Does LightOnOCR's native output capture bold, italic, underline? — sample 2

Companion to `investigation-docling-native-json-sample2.ipynb`. Docling had three
levels to look at (JSON, Markdown, page cells). LightOnOCR has **one**: it is a
vision-language model that looks at a picture of the page and writes text, and that
text is the whole output. There is no data structure underneath it to inspect.

So the question becomes: what formatting does the model write **on its own**, and
what does it write **when asked** for the project's markers? Two runs on the same
page images:

| run | prompt | what it shows |
|---|---|---|
| native | "Transcribe all text on this page exactly as it appears, in reading order." — no mention of formatting | the model's own habits |
| project | the extractor's `_TRANSCRIBE_PROMPT`, which asks for `**bold**`, `_italic_`, `++underline++` | what the pipeline actually gets |

Sample 2 is the test document because pdfplumber, reading the PDF's font data,
finds **42 bold, 11 underlined and 41 italic** runs in it. Decoding is greedy, so both
runs are deterministic and the project run should reproduce the cached stage-1 text.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import collections
import json
import logging
import re
import time
import warnings

os.environ['TQDM_DISABLE'] = '1'
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

import pandas as pd
import torch
from IPython.display import display

pd.set_option('display.max_colwidth', 80)

SAMPLE = 2
PDF = Path(f'data/input/pdfs/sample{SAMPLE}.pdf')

from dmpbridge.extractors import get_extractor
from dmpbridge.extractors import lighton_extractor as L

t0 = time.time()
ex = get_extractor('lightonocr')
print(f'model {L.DEFAULT_MODEL_ID} loaded in {time.time() - t0:.0f} s, '
      f'{torch.cuda.memory_allocated() / 1024**3:.1f} GB VRAM')

pages = ex._render_pages(PDF)
print(f'{len(pages)} pages rendered at {L._RENDER_DPI} dpi, {pages[0].size[0]}x{pages[0].size[1]} px')


def ocr(image, prompt):
    """The extractor's _ocr_image, with the prompt as a parameter."""
    messages = [{'role': 'user', 'content': [{'type': 'image'},
                                             {'type': 'text', 'text': prompt}]}]
    text_prompt = ex._processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = ex._processor(text=text_prompt, images=image, return_tensors='pt').to(ex._device)
    with torch.no_grad():
        out = ex._model.generate(**inputs, max_new_tokens=ex._max_new_tokens)
    return ex._processor.decode(out[0, inputs['input_ids'].shape[-1]:], skip_special_tokens=True)


NATIVE_PROMPT = ('Transcribe all text on this document page exactly as it appears, '
                 'in reading order.')
PROJECT_PROMPT = L._TRANSCRIBE_PROMPT
print()
print('project prompt:', PROJECT_PROMPT)


model lightonai/LightOnOCR-2-1B loaded in 50 s, 1.9 GB VRAM
5 pages rendered at 150 dpi, 1275x1650 px

project prompt: Transcribe all text on this document page exactly as it appears, in reading order. Preserve the source formatting using these markers: wrap visually emphasized text (bold or larger than the body text) in **double asterisks**, italic text in _underscores_, and underlined text in ++double plus signs++. Do not use these markers for anything that isn't actually formatted that way in the source.


In [2]:
runs = {}
for name, prompt in (('native', NATIVE_PROMPT), ('project', PROJECT_PROMPT)):
    t0 = time.time()
    runs[name] = '\n\n'.join(ocr(img, prompt) for img in pages)
    print(f'{name:8s} {len(runs[name]):6d} chars   {time.time() - t0:5.1f} s')

cached = json.loads(Path(f'data/output/1_extracted/lightonocr/sample{SAMPLE}.json')
                    .read_text(encoding='utf-8'))[0]['text']
print()
print('project run reproduces cached stage 1 exactly:', runs['project'] == cached)


native    14599 chars    63.7 s


project   14857 chars    64.1 s

project run reproduces cached stage 1 exactly: True


## 1. What markup each run contains

Counted directly in the generated text. `#` headings and `**bold**` are the model's
two ways of showing emphasis; `*italic*` is Markdown's italic; `_italic_` and
`++underline++` are the project's requested forms.


In [3]:
def markup(t):
    return {
        '# headings':      len(re.findall(r'^#+ ', t, re.M)),
        '**bold**':        len(re.findall(r'\*\*[^*\n]+?\*\*', t)),
        '*italic*':        len(re.findall(r'(?<![*\w])\*(?!\*)[^*\n]+?\*(?![*\w])', t)),
        '_italic_':        len(re.findall(r'(?<!\w)_[^_\n]+?_(?!\w)', t)),
        '++underline++':   len(re.findall(r'\+\+[^+\n]+?\+\+', t)),
        '<u>':             t.count('<u>'),
    }

table = pd.DataFrame({name: markup(t) for name, t in runs.items()})
table['PDF has (pdfplumber)'] = pd.Series({'**bold**': '42', '_italic_': '41', '++underline++': '11'})
display(table.fillna(''))


,native,project,PDF has (pdfplumber)
# headings,8,8,
**bold**,7,7,42
*italic*,0,1,
_italic_,0,0,41
++underline++,0,0,11
<u>,0,0,


The opening of each run, so the difference is visible rather than counted.


In [4]:
for name, t in runs.items():
    print(f'───── {name} ─────')
    print('\n'.join(t.splitlines()[:14]))
    print()


───── native ─────
# Center for Bio-Inspired Energy Science

## 1. Data sharing and preservation

Data management plans should describe whether and how data generated in the course of the proposed research will be **shared** and **preserved**. If the plan is not to share and/or preserve certain data, then the plan must explain the basis of the decision (for example, cost/benefit considerations, other parameters of feasibility, scientific appropriateness, or limitations discussed in #4). At a minimum, DMPs must describe how data sharing and preservation will enable **validation** of results, or how results could be validated if data are not shared or preserved.

### Roles & Responsibilities

For the proposed research, Director Samuel Stupp with help from the Executive Director of Research will take the lead and responsibility for coordinating and ensuring data storage and access and communicating expectations to all investigators. However, all senior investigators will also be involved 

## 2. Reference — the runs the PDF actually contains

pdfplumber's stage-1 text for the same document, with its font-derived markers.
Each of its bold / underline / italic runs is then looked up in both LightOnOCR
outputs to see how the model rendered that exact text.


In [5]:
stage1 = json.loads(Path(f'data/output/1_extracted/pdfplumber/sample{SAMPLE}.json')
                    .read_text(encoding='utf-8'))[0]['text']
ref = {
    'bold':      re.findall(r'\*\* (.+?) \*\*', stage1),
    'underline': re.findall(r'\+\+ (.+?) \+\+', stage1),
    'italic':    re.findall(r'(?<![\w*+])_ (.+?) _(?![\w])', stage1),
}
print({k: len(v) for k, v in ref.items()})


def clean(s):
    return re.sub(r'\s+', ' ', re.sub(r'[*_+#]', '', s)).strip()


def marked_spans(text):
    """Every span the model marked, as (marker, cleaned text)."""
    out = []
    for line in text.splitlines():
        m = re.match(r'^(#+) (.*)', line)
        if m:
            out.append(('heading ' + m.group(1), clean(m.group(2))))
    for pat, tag in ((r'\*\*(.+?)\*\*', '**bold**'),
                     (r'(?<![*\w])\*(?!\*)(.+?)\*(?![*\w])', '*italic*'),
                     (r'(?<!\w)_(.+?)_(?!\w)', '_italic_'),
                     (r'\+\+(.+?)\+\+', '++underline++')):
        out.extend((tag, clean(s)) for s in re.findall(pat, text))
    return out


def rendered_as(text, spans, phrase):
    """How the model rendered pdfplumber's *phrase*: which marked span, if any,
    contains it (or sits inside it), else plain. Trailing punctuation is ignored
    because the model often drops the period after a label."""
    key = clean(phrase).rstrip('.:;,')[:28]
    if key not in clean(text):
        return 'not found'
    full = clean(phrase).rstrip('.:;,')
    # A span counts if it contains the phrase, or if it IS the phrase (covers most
    # of it) — one bolded word inside a long plain line does not count.
    tags = sorted({tag for tag, s in spans
                   if key in s or (s in full and len(s) >= 0.6 * len(full))})
    return ', '.join(tags) or 'plain'


spans = {name: marked_spans(t) for name, t in runs.items()}
rows = []
for kind, phrases in ref.items():
    for p in phrases:
        rows.append({'signal in PDF': kind, 'text': p[:60],
                     'native run': rendered_as(runs['native'], spans['native'], p),
                     'project run': rendered_as(runs['project'], spans['project'], p)})
per_run = pd.DataFrame(rows)
display(per_run[per_run['signal in PDF'] == 'bold'].head(16))
display(per_run[per_run['signal in PDF'] == 'underline'])
display(per_run[per_run['signal in PDF'] == 'italic'].head(8))


{'bold': 42, 'underline': 11, 'italic': 41}


,signal in PDF,text,native run,project run
0,bold,Center for Bio-Inspired Energy Science,heading #,heading #
1,bold,1. Data sharing and preservation,heading ##,heading ##
2,bold,Data management plans should describe whether and how data g,plain,plain
3,bold,of the proposed research will be ++ shared ++ and ++ preserv,plain,plain
4,bold,"preserve certain data, then the plan must explain the basis",plain,plain
5,bold,"cost/benefit considerations, other parameters of feasibility",plain,plain
6,bold,"limitations discussed in #4). At a minimum, DMPs must descri",plain,plain
7,bold,"preservation will enable ++ validation ++ of results, or how",plain,plain
8,bold,are not shared or preserved.,plain,plain
9,bold,Roles & Responsibilities.,heading ###,heading ###


,signal in PDF,text,native run,project run
42,underline,shared,**bold**,**bold**
43,underline,preserved.,**bold**,**bold**
44,underline,validation,**bold**,**bold**
45,underline,Principles,plain,plain
46,underline,description of data management resources,plain,plain
47,underline,additional guidance from the sponsporing program.,plain,plain
48,underline,description of data management resources,plain,plain
49,underline,additional guidance,plain,plain
50,underline,from the sponsoring program.,plain,plain
51,underline,Personally,plain,plain


,signal in PDF,text,native run,project run
53,italic,"A brief, high-level description of the data to be generated",plain,plain
54,italic,the course of the proposed research and which of these are c,plain,plain
55,italic,necessary to Validate the research findings.,plain,plain
56,italic,A statement of plans for data and metadata content and forma,plain,plain
57,italic,"applicable, a description of documentation plans, annotation",plain,plain
58,italic,"the selection of appropriate standards. (Existing, accepted",plain,plain
59,italic,possible. Where community standards are missing or inadequat,plain,plain
60,italic,"strategies that facilitate data sharing, and should advise t",plain,plain


Summed over every run in the PDF: how many did each LightOnOCR output mark **at all**
(with any marker), and how many with the **right** marker.


In [6]:
RIGHT = {'bold': ('**bold**', 'heading'), 'italic': ('*italic*', '_italic_'),
         'underline': ('++underline++',)}

summary = []
for kind in ref:
    sub = per_run[per_run['signal in PDF'] == kind]
    for run in ('native run', 'project run'):
        marked = sub[run].apply(lambda v: v not in ('plain', 'not found'))
        right  = sub[run].apply(lambda v: any(r in v for r in RIGHT[kind]))
        summary.append({'signal': kind, 'run': run.split()[0], 'in PDF': len(sub),
                        'marked at all': int(marked.sum()),
                        'with the right marker': f'{int(right.sum())}  ({"/".join(RIGHT[kind])})',
                        'not found in output': int((sub[run] == 'not found').sum())})
display(pd.DataFrame(summary))


,signal,run,in PDF,marked at all,with the right marker,not found in output
0,bold,native,42,12,12 (**bold**/heading),1
1,bold,project,42,12,12 (**bold**/heading),1
2,underline,native,11,3,0 (++underline++),0
3,underline,project,11,3,0 (++underline++),0
4,italic,native,41,1,0 (*italic*/_italic_),0
5,italic,project,41,1,0 (*italic*/_italic_),0


## 3. Things the model wrote that are not in the PDF

A transcriber that generates text can also add text. Lines in the output whose
words do not appear anywhere in pdfplumber's extraction.


In [7]:
pdf_words = set(re.findall(r'[a-z]{4,}', stage1.lower()))
for name, t in runs.items():
    extra = []
    for line in t.splitlines():
        words = re.findall(r'[a-z]{4,}', line.lower())
        if len(words) >= 4 and sum(w not in pdf_words for w in words) >= len(words) // 2:
            extra.append(line.strip()[:110])
    print(f'{name}: {len(extra)} line(s) not grounded in the PDF')
    for e in extra[:5]:
        print('   ', repr(e))


native: 0 line(s) not grounded in the PDF
project: 1 line(s) not grounded in the PDF
    '*Note: The above Markdown preserves the original structure and content of the document image, including the pa'


## 4. Conclusion

LightOnOCR's native output is prose in Markdown, and its idea of formatting is
Markdown's: headings and bold for anything that looks emphasised, `*…*` for the
occasional italic, and no notion of underline at all — Markdown has none, so the
model never learned one.

- **The prompt changes almost nothing.** The native and project runs contain the
  same headings and the same `**` spans; the marker instructions added one
  `*italic*` and a line of commentary that is not in the PDF (section 3). The
  model transcribes the way it was trained to, and the request for `_` and `++` is
  simply not followed.
- **Bold is reported by shape, not measured.** Short bold labels on their own line
  become `#` headings; the bold *paragraph* at the top of the document comes back
  plain — the model bolds the few underlined words inside it instead. So of the
  42 bold runs, only the ones that look like headings are marked (section 2).
- **Italic is essentially lost.** The 41 italic runs are whole instruction
  paragraphs — the text that marks `section.description` — and they come back as
  ordinary body text in both runs.
- **Underline is never produced.** The model has no marker for it; an underlined
  word is shown as `**bold**` or not at all.

Unlike Docling, there is no lower level to recover from: the model's text is the
only artefact, and the prompt is the only control — and section 1 shows the
prompt does not move the result. This is the nature of the approach, not a
configuration problem. The extractor's docstring already records that the model
ignores the `++` instruction; this notebook measures the same thing for all three
markers, against the PDF's actual font data.
